In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126


Looking in indexes: https://download.pytorch.org/whl/cu126


In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 73.2 MB/s eta 0:00:00


In [ ]:
!uv pip install surya-ocr


Using Python 3.12.12 environment at: /usr
Resolved 61 packages in 507ms
Prepared 11 packages in 770ms
Uninstalled 2 packages in 15ms
Installed 11 packages in 17ms
 + cfgv==3.5.0
 + distlib==0.4.0
 + filetype==1.2.0
 + identify==2.6.15
 + nodeenv==1.9.1
 - opencv-python-headless==4.12.0.88
 + opencv-python-headless==4.11.0.86
 - pillow==11.3.0
 + pillow==10.4.0
 + pre-commit==4.5.0
 + pypdfium2==4.30.0
 + surya-ocr==0.17.0
 + virtualenv==20.35.4


In [ ]:
!uv pip install pypdfium2

Using Python 3.12.12 environment at: /usr
Audited 1 package in 100ms


# extract folder of files

In [ ]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
import pypdfium2 as pdfium
import os


In [ ]:
# Initialize predictors once (outside the loop for efficiency)
foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

In [ ]:
input_folder = "/content/drive/MyDrive/Graduation Project/Books"
output_folder = "/content/drive/MyDrive/Graduation Project/Collected Data"

pdf_files = [f for f in os.listdir(input_folder) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files to process\n")


Found 6 PDF files to process



In [ ]:
# Process each PDF file
for file_index, pdf_filename in enumerate(pdf_files, 1):
    pdf_path = os.path.join(input_folder, pdf_filename)

    # Create output filename (replace .pdf with .txt)
    output_filename = pdf_filename.replace('.pdf', '.txt')
    output_path = os.path.join(output_folder, output_filename)

    print(f"\n{'='*60}")
    print(f"[{file_index}/{len(pdf_files)}] Processing: {pdf_filename}")
    print(f"{'='*60}")

    try:
        # Open PDF
        pdf = pdfium.PdfDocument(pdf_path)
        n_pages = len(pdf)
        print(f"Total Pages: {n_pages}")
        print("Starting text extraction...\n")

        # Store all extracted text
        all_text = []

        # Loop through all pages (or first 35 if you want)
        for page_num in range(n_pages):
            try:
                print(f"  Processing page {page_num + 1}/{n_pages}...", end=" ")

                # Render page
                page = pdf[page_num]
                bitmap = page.render(scale=1, rotation=0)
                pil_image = bitmap.to_pil()

                # Step 1: Detect all text boxes on the page
                detections = detection_predictor([pil_image])
                detected_boxes = detections[0].bboxes

                # Step 2: Recognize text in each detected box
                if len(detected_boxes) > 0:
                    predictions = recognition_predictor([pil_image], det_predictor=detection_predictor)
                    page_text_lines = [predictions[0].text_lines[i].text for i in range(len(predictions[0].text_lines))]
                else:
                    page_text_lines = []

                # Store text from this page
                all_text.append({
                    'page': page_num + 1,
                    'full_text': '\n'.join(page_text_lines)
                })

                print(f"({len(detected_boxes)} boxes, {len(page_text_lines)} lines)")

            except Exception as e:
                print(f"Error: {str(e)}")
                continue

        # Save to file
        with open(output_path, 'w', encoding='utf-8') as f:
            for page_data in all_text:
                f.write(f"\n{'='*50}\n")
                f.write(f"PAGE {page_data['page']}\n")
                f.write(f"{'='*50}\n")
                f.write(page_data['full_text'])
                f.write("\n")

        print(f"\n Extraction complete! Processed {len(all_text)} pages")
        print(f"Text saved to: {output_path}")

    except Exception as e:
        print(f"\n✗ Error processing file {pdf_filename}: {str(e)}")
        continue

print(f"\n{'='*60}")
print("✓ All files processed successfully!")
print(f"{'='*60}")


[1/6] Processing: قانون_رقم_7_لسنة_2025_بتعديل_بعض_أحكام_قانون_الإجراءات_الضريبية.pdf
Total Pages: 2
Starting text extraction...

  Processing page 2/2... 

Recognizing Text: 100%|██████████| 16/16 [00:03<00:00,  5.21it/s]


(16 boxes, 16 lines)

 Extraction complete! Processed 1 pages
Text saved to: /content/drive/MyDrive/Graduation Project/Collected Data/قانون_رقم_7_لسنة_2025_بتعديل_بعض_أحكام_قانون_الإجراءات_الضريبية.txt

[2/6] Processing: الموسوعةالشاملة_في_شرح_الشهر_العقاري_والسجل_العيني_5.pdf
Total Pages: 840
Starting text extraction...

  Processing page 2/840... 

Recognizing Text: 100%|██████████| 6/6 [00:00<00:00, 17.48it/s]


(6 boxes, 6 lines)
  Processing page 3/840... 

...

(47 boxes, 47 lines)
  Processing page 701/702... 

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.74it/s]


(0 boxes, 0 lines)
  Processing page 702/702... 

Recognizing Text: 100%|██████████| 10/10 [00:00<00:00, 11.82it/s]

(10 boxes, 10 lines)

 Extraction complete! Processed 701 pages
Text saved to: /content/drive/MyDrive/Graduation Project/Collected Data/سن_وصياغة_التشريعات_الكتاب_الاول_ج٢.txt

✓ All files processed successfully!


# extract single file

In [ ]:
# pdf_path ="/content/كل_ما_يخص_ايصالات_الامانة.pdf"

# pdf = pdfium.PdfDocument(pdf_path)
# n_pages = len(pdf)

# print(f"Total Pages: {n_pages}")
# print("Starting text extraction...\n")



# # Store all extracted text
# all_text = []

# # Loop through first 35 pages only
# for page_num in range(1,n_pages):
#     try:
#         print(f"Processing page {page_num + 1}/{n_pages}...")

#         # Render page
#         page = pdf[page_num]
#         bitmap = page.render(
#             scale=1,
#             rotation=0,
#         )
#         pil_image = bitmap.to_pil()

#         # Step 1: Detect all text boxes on the page
#         detections = detection_predictor([pil_image])
#         detected_boxes = detections[0].bboxes  # Get all bounding boxes

#         print(f"  Found {len(detected_boxes)} text boxes")

#         # Step 2: Recognize text in each detected box
#         if len(detected_boxes) > 0:
#             predictions = recognition_predictor([pil_image], det_predictor=detection_predictor)
#             page_text_lines = [predictions[0].text_lines[i].text for i in range(len(predictions[0].text_lines))]
#         else:
#             page_text_lines = []

#         # Store text from this page
#         all_text.append({
#             'page': page_num + 1,
#             'full_text': '\n'.join(page_text_lines)
#         })

#         print(f"  ✓ Extracted {len(page_text_lines)} text lines from {len(detected_boxes)} boxes")

#     except Exception as e:
#         print(f"  ✗ Error processing page {page_num + 1}: {str(e)}")
#         continue

# print(f"\n{'='*50}")
# print(f"Extraction complete! Processed {len(all_text)} pages")
# print(f"{'='*50}\n")

# # Print results
# for page_data in all_text:
#     print(f"\n--- PAGE {page_data['page']}")
#     print(page_data['full_text'])
#     print("-" * 50)

# # Optional: Save to file
# output_file = "/content/text2.txt"
# with open(output_file, 'w', encoding='utf-8') as f:
#     for page_data in all_text:
#         f.write(f"\n{'='*50}\n")
#         f.write(f"PAGE {page_data['page']}\n")
#         f.write(f"{'='*50}\n")
#         f.write(page_data['full_text'])
#         f.write("\n")

# print(f"\nText saved to '{output_file}'")

Total Pages: 16
Starting text extraction...

Processing page 2/16...


Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]


  Found 42 text boxes


Recognizing Text: 100%|██████████| 42/42 [00:04<00:00, 10.18it/s]


  ✓ Extracted 42 text lines from 42 boxes
Processing page 3/16...


...

  ✓ Extracted 40 text lines from 40 boxes
Processing page 16/16...


Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  6.14it/s]


  Found 7 text boxes


Recognizing Text: 100%|██████████| 7/7 [00:02<00:00,  3.26it/s]

  ✓ Extracted 7 text lines from 7 boxes

Extraction complete! Processed 15 pages


--- PAGE 2

    ٢- توقيت كتابة وصل الأمانة لازم يكون هو نفسه توقيت التوقيع أو بصمة المدين.

٣- متحاولش تكتب تاريخ للوصل يعني متكتبش تم تحريره في كذا عشان متدخلش في جدل هل الدعوى الجنائية
بتنقضي بمرور ثلاث سنوات من تاريخ طلب الوصل ولا من تاريخ تحريره فيفضل عدم كتابة تاريخ تحريره.
٤- لازم تتأكد كويس جدا من عنوان ومحل إقامة صاحب التوقيع وخلى بالك من تاريخ البطاقة لان ممكن تكون
منتهية وتم تغيير محل للإقامة اللي فيها.
٥-لازم يكون التوقيع بتاعه واسمه بشكل طبيعي وواضح وضمانا بشكل أكبر من الممكن تاخد بصمته بجوار
التوقيع.
كيفية ملئ بيانات ايصال الأمانة:
ايصال أمات
CLA
استلمت أنا الموهع أدناه /
(500
المقيم / -
( الله عدورها / ( 3 )
بطاقة رقم / -
(0)
من السيد /
ميلغ وقدره
وذلك بصفة أمانة لتسليمها إلى السيد / .....
وإذا لم أقم بتسليم هذا البلغ أعتبر كانتنا للأمانة وأعاقب فانونيا
المقريما هيه ، (٩) السعة ، (٩)
فيه دفاتر بتتباع في المكتبات ، فيها ايصالات مطبوعه ﴿ ﴿ اللَّمِ فِي الصَّورِهِ ﴾ وتقدر برضه تكتبه في ورقه
عاد